In [0]:
from pyspark.sql.functions import col, when, regexp_replace, to_date, lit, trim

# Load bronze customers data
bronze_customers = spark.table("novocart.bronze.customers_raw")

# Standardize country_code (multilingual, special chars, encoding)
bronze_customers = bronze_customers.withColumn(
    "country_code",
    trim(regexp_replace(col("country_code"), "[^A-Za-z]", ""))
)

# Standardize customer_name (remove special chars, encoding issues)
bronze_customers = bronze_customers.withColumn(
    "customer_name",
    regexp_replace(col("customer_name"), "[^A-Za-z0-9\s]", "")
)

# Standardize registration_date (inconsistent formats)
bronze_customers = bronze_customers.withColumn(
    "registration_date",
    try_to_date(col("registration_date"), "yyyy-MM-dd")
)

# Handle non-standard nulls in email
bronze_customers = bronze_customers.withColumn(
    "email",
    when(col("email").isin("\\N", "?", ""), lit(None)).otherwise(col("email"))
)

# Handle non-standard nulls in channel
bronze_customers = bronze_customers.withColumn(
    "channel",
    when(col("channel").isin("\\N", "?", ""), lit(None)).otherwise(col("channel"))
)

# Remove orphan records (invalid customer_id)
bronze_customers = bronze_customers.filter(col("customer_id").isNotNull())

# Drop first four column

bronze_customers=bronze_customers.drop("_file","_line","_modified","_fivetran_synced")

# Store cleaned data in silver
bronze_customers.write.format("delta").mode("overwrite").saveAsTable("novocart.silver.customers_cleaned")

display(spark.table("novocart.silver.customers_cleaned"))

In [0]:
from pyspark.sql.functions import col, when, regexp_replace, to_date, lit, trim

# Load bronze orders data
bronze_orders = spark.table("novocart.bronze.orders_raw")

# Standardize country_code (multilingual, special chars, encoding)
bronze_orders = bronze_orders.withColumn(
    "country_code",
    trim(regexp_replace(col("country_code"), "[^A-Za-z]", ""))
)

# Standardize channel (remove non-standard nulls)
bronze_orders = bronze_orders.withColumn(
    "channel",
    when(col("channel").isin("\\N", "?", ""), lit(None)).otherwise(col("channel"))
)

# Standardize order_date (inconsistent formats)
bronze_orders = bronze_orders.withColumn(
    "order_date",
    try_to_date(col("order_date"), "yyyy-MM-dd")
)

# Remove orphan records (invalid order_id)
bronze_orders = bronze_orders.filter(col("order_id").isNotNull())

# Drop first four columns
bronze_orders = bronze_orders.drop("_file", "_line", "_modified", "_fivetran_synced")

# Store cleaned data in silver
bronze_orders.write.format("delta").mode("overwrite").saveAsTable("novocart.silver.orders_cleaned")

display(spark.table("novocart.silver.orders_cleaned"))